# Module 21 — Cost, Latency & Agent Economics

> **SDKs:** `dataclasses`, `statistics`

| Part | Topic |
|------|-------|
| **1** | Caching Strategies — Semantic vs Exact Match |
| **2** | Model Cascades — Quality/Cost Tradeoffs |
| **3** | Latency Budgets — Parallel vs Sequential Execution |


---
## Part 1 — Semantic Caching

Using embeddings to cache similar agent queries can reduce costs by 40-60%.

In [1]:
class SemanticCache:
    def __init__(self):
        self.cache = {
            "reset password": "To reset your password, visit /forgot-password.",
            "where is my order": "Track your order at /tracking with your order ID.",
        }
    
    def query(self, text: str) -> tuple[bool, str]:
        # Simulated semantic match
        if "password" in text.lower() or "login" in text.lower():
            return True, self.cache["reset password"]
        return False, "CACHE MISS"

cache = SemanticCache()
print("💾  Semantic Caching Demo")
print("=" * 60)

queries = [
    "I forgot my password",
    "How do I login?",
    "Can I return my shoes?",
]

for q in queries:
    hit, result = cache.query(q)
    icon = "✅ HIT" if hit else "❌ MISS"
    print(f"  [{icon}] '{q}' -> {result[:40]}")


💾  Semantic Caching Demo
  [✅ HIT] 'I forgot my password' -> To reset your password, visit /forgot-pa
  [✅ HIT] 'How do I login?' -> To reset your password, visit /forgot-pa
  [❌ MISS] 'Can I return my shoes?' -> CACHE MISS


---
## Part 2 — Cost Analysis of Multi-Agent Architectures

Adding agents multiplies the 'Multi-Agent Tax' through context window repetition.

In [2]:
def calculate_cost(workflow_type: str, steps: int) -> float:
    base_prompt_tokens = 2000
    output_tokens_per_step = 500
    cost_per_1k_in = 0.005
    cost_per_1k_out = 0.015
    
    total_cost = 0.0
    if workflow_type == "Single Agent":
        # Context grows with each step
        for i in range(steps):
            in_tokens = base_prompt_tokens + (i * output_tokens_per_step)
            total_cost += (in_tokens / 1000) * cost_per_1k_in
            total_cost += (output_tokens_per_step / 1000) * cost_per_1k_out
    elif workflow_type == "Multi-Agent Cascade":
        # Each agent gets a fresh, but larger, handoff context
        for i in range(steps):
            in_tokens = base_prompt_tokens + (i * output_tokens_per_step * 2) # Overlap penalty
            total_cost += (in_tokens / 1000) * cost_per_1k_in
            total_cost += (output_tokens_per_step / 1000) * cost_per_1k_out
            
    return total_cost

print("💸  Agent Economics Demo")
print("=" * 60)
for steps in [3, 5, 10]:
    c_single = calculate_cost("Single Agent", steps)
    c_multi  = calculate_cost("Multi-Agent Cascade", steps)
    print(f"  {steps:2d} Steps | Single: ${c_single:.3f} | Multi: ${c_multi:.3f} | Tax: +{((c_multi-c_single)/c_single)*100:.0f}%")


💸  Agent Economics Demo
   3 Steps | Single: $0.060 | Multi: $0.068 | Tax: +12%
   5 Steps | Single: $0.113 | Multi: $0.138 | Tax: +22%
  10 Steps | Single: $0.288 | Multi: $0.400 | Tax: +39%
